# S5-nano (Maritaca AI) no ENEM 2026, 1º dia

O S5-nano abriu os pesos em 07/11/2026 e a prova do 1º dia do ENEM 2026 foi aplicada em 08/11/2026, então o modelo nunca viu essas questões.
O modelo recebe as **páginas escaneadas do caderno azul como imagem** e responde as 90 questões (45 de Linguagens, 45 de Ciências Humanas; nas questões 01 a 05, a opção de inglês).

- Prova: caderno azul publicado pelo Diário do Nordeste (cópia idêntica hospedada pelo FII Brasil).
- Gabarito: **extraoficial** da Plataforma Assaad (o oficial do INEP sai até o fim de novembro). Na questão 48 o gabarito aceita C ou D.

**Antes de rodar:** Ambiente de execução → Alterar o tipo de ambiente → **GPU T4**. Depois, Ambiente de execução → Executar tudo.

In [ ]:
!pip -q install -U transformers accelerate pymupdf pytesseract
!apt-get -qq install -y tesseract-ocr tesseract-ocr-por > /dev/null
!curl -sSfL -A "Mozilla/5.0" -o prova_azul_d1.pdf "https://fiibrasil.com/wp-content/plugins/fiibrasil-gabarito-enem-2026/assets/prova-azul-enem-2026.pdf"

import re, json, time, os
import pymupdf, pytesseract, torch
from PIL import Image

# Gabarito extraoficial, Plataforma Assaad, caderno azul, 1º dia (01-05 = inglês)
GAB = ("BAEDC" "DBCABDBDAB" "EEDBEABCDCECCEA" "ACADCBABABCDEDE"
       "CC?DEBEDDDEAECB" "AADCABBAEAACDED" "BEDABDEECBEACAE")
GAB = {i + 1: g for i, g in enumerate(GAB)}
GAB[48] = "CD"  # o gabarito extraoficial aceita C ou D
assert len(GAB) == 90

# Páginas em PNG (150 dpi) e, com OCR, em que página e posição aparece cada "QUESTÃO NN"
doc = pymupdf.open("prova_azul_d1.pdf")
os.makedirs("paginas", exist_ok=True)
heads = {}  # n -> (pagina, coluna, y)
for p in range(doc.page_count):
    fn = f"paginas/p{p:02d}.png"
    doc[p].get_pixmap(dpi=150).save(fn)
    img = Image.open(fn)
    d = pytesseract.image_to_data(img, lang="por", output_type=pytesseract.Output.DICT)
    for i, w in enumerate(d["text"]):
        if re.fullmatch(r"QUEST[ÃA]O", w.strip()):
            nxt = " ".join(d["text"][i + 1:i + 3])
            m = re.match(r"\s*0?(\d{1,2})\b", nxt)
            if m and 1 <= int(m.group(1)) <= 90 and int(m.group(1)) not in heads:  # 1ª vez = inglês em 01-05
                col = int(d["left"][i] > img.width / 2)
                heads[int(m.group(1))] = (p, col, d["top"][i])
faltando = [n for n in range(1, 91) if n not in heads]
print(f"{doc.page_count} páginas, {len(heads)}/90 questões localizadas. Faltando: {faltando}")

# Questões não achadas pelo OCR ficam com as páginas das vizinhas
def paginas_da(n):
    if n in heads:
        p = heads[n][0]
        mesma = sorted((c, y, k) for k, (pp, c, y) in heads.items() if pp == p)
        ultima = mesma[-1][2] == n
        return [p, p + 1] if ultima and p + 1 < doc.page_count else [p]
    antes = max((heads[k][0] for k in heads if k < n), default=1)
    depois = min((heads[k][0] for k in heads if k > n), default=doc.page_count - 1)
    return list(range(antes, depois + 1))

PAGS = {n: paginas_da(n) for n in range(1, 91)}
print({n: PAGS[n] for n in (1, 5, 6, 45, 46, 90)})

In [ ]:
from transformers import AutoProcessor, AutoModelForMultimodalLM

MODEL = "maritaca-ai/S5-nano"
# A T4 não tem bf16 nativo; se as respostas saírem estranhas no teste rápido, troque para torch.bfloat16
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
processor = AutoProcessor.from_pretrained(MODEL)
model = AutoModelForMultimodalLM.from_pretrained(MODEL, dtype=DTYPE, device_map="auto")
print(torch.cuda.get_device_name(), DTYPE)

PROMPT = ("Você está fazendo a prova do ENEM 2026. As imagens são páginas do caderno de questões. "
          "Responda à QUESTÃO {n}{lingua}. Explique o raciocínio em poucas linhas e termine com uma linha "
          "no formato 'Resposta: X', em que X é A, B, C, D ou E.")

def responder(n, pensar=False, max_new_tokens=1024):
    lingua = " (a versão de inglês, não a de espanhol)" if n <= 5 else ""
    content = [{"type": "image", "path": f"paginas/p{p:02d}.png"} for p in PAGS[n]]
    content.append({"type": "text", "text": PROMPT.format(n=f"{n:02d}", lingua=lingua)})
    inputs = processor.apply_chat_template(
        [{"role": "user", "content": content}], add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt", enable_thinking=pensar).to(model.device)
    sampling = dict(temperature=1.0, top_p=0.95) if pensar else dict(temperature=0.7, top_p=0.8)
    torch.manual_seed(0)
    t0 = time.time()
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, top_k=20, **sampling)
    gen = out[0][inputs["input_ids"].shape[-1]:]
    texto = processor.decode(gen, skip_special_tokens=True)
    final = texto.split("</think>")[-1]
    m = re.findall(r"Resposta:\s*\**\(?([A-E])\b", final)
    letra = m[-1] if m else None
    return {"questao": n, "area": "Linguagens" if n <= 45 else "Humanas", "paginas": PAGS[n],
            "gabarito": GAB[n], "resposta": letra, "certa": bool(letra and letra in GAB[n]),
            "segundos": round(time.time() - t0, 1), "tokens": len(gen), "texto": texto[-1500:]}

def rodar(questoes, pensar=False, max_new_tokens=1024):
    res = []
    for n in questoes:
        r = responder(n, pensar, max_new_tokens)
        res.append(r)
        print(f"Q{n:02d} gabarito {r['gabarito']} resposta {r['resposta']} {'✔' if r['certa'] else '✘'} "
              f"({r['segundos']} s, {r['tokens']} tokens)", flush=True)
    return res

def resumo(res, nome):
    ok = sum(r["certa"] for r in res)
    print(f"\n{nome}: {ok}/{len(res)} ({100 * ok / len(res):.1f}%)")
    for a in ("Linguagens", "Humanas"):
        rs = [r for r in res if r["area"] == a]
        if rs:
            print(f"  {a}: {sum(r['certa'] for r in rs)}/{len(rs)}")
    print(f"  sem resposta: {sum(r['resposta'] is None for r in res)}, "
          f"média {sum(r['segundos'] for r in res) / len(res):.1f} s por questão")

In [ ]:
# Teste rápido: 3 questões. Confira se o texto faz sentido antes de rodar tudo.
teste = rodar([1, 46, 90])
print(teste[0]["texto"])

In [ ]:
# Modo Instruct (resposta direta), 90 questões. Na T4 leva por volta de 30 a 60 minutos.
instruct = rodar(range(1, 91), pensar=False)
resumo(instruct, "S5-nano Instruct")
print("COMPACTO", json.dumps({r["questao"]: [r["resposta"], int(r["certa"]), r["segundos"]] for r in instruct}))
json.dump({"modelo": MODEL, "modo": "instruct", "gabarito": "extraoficial Plataforma Assaad", "resultados": instruct},
          open("resultado_s5nano_enem2026_d1_instruct.json", "w"), ensure_ascii=False, indent=1)

In [ ]:
# Salva no seu Google Drive (pasta linkedin-resultados)
from google.colab import drive
import shutil
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/linkedin-resultados', exist_ok=True)
for f in os.listdir('.'):
    if f.startswith('resultado_s5nano'):
        shutil.copy(f, '/content/drive/MyDrive/linkedin-resultados/' + f)
        print('Salvo em Meu Drive/linkedin-resultados/' + f)

### Opcional: modo Think

In [ ]:
# Opcional, se sobrar tempo: modo Think (raciocina antes), que é bem mais lento.
# Comece com LIMITE = 15 para ver o tempo por questão; use None para as 90.
LIMITE = 15
qs = list(range(1, 91))[:LIMITE] if LIMITE else range(1, 91)
think = rodar(qs, pensar=True, max_new_tokens=8192)
resumo(think, "S5-nano Think")
print("COMPACTO", json.dumps({r["questao"]: [r["resposta"], int(r["certa"]), r["segundos"]] for r in think}))
json.dump({"modelo": MODEL, "modo": "think", "gabarito": "extraoficial Plataforma Assaad", "resultados": think},
          open("resultado_s5nano_enem2026_d1_think.json", "w"), ensure_ascii=False, indent=1)

In [ ]:
# Salva no seu Google Drive (pasta linkedin-resultados)
from google.colab import drive
import shutil
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/linkedin-resultados', exist_ok=True)
for f in os.listdir('.'):
    if f.startswith('resultado_s5nano'):
        shutil.copy(f, '/content/drive/MyDrive/linkedin-resultados/' + f)
        print('Salvo em Meu Drive/linkedin-resultados/' + f)